# Run the architecture-conformance experiments on a free GPU (Kaggle or Colab)

This notebook runs the **same code** as the project on a free cloud GPU with Ollama, then packages the results for download.

**Current goal: the HARD TIER run (section 6).** Sections 1-5 are one-time setup; run them in order every new session.
The "Later" section at the end holds the confirmatory runs (primary, minimal prompt, held-out wording). Do not run those yet.

**Before you start**
1. **Kaggle:** create a notebook, set *Accelerator* to a GPU, turn *Internet* on. Add the project zip (any file named `architecture-analyzer*.zip`; use the Kaggle zip, which has no `frontend/` folder because Kaggle rejects `__name__` folders) as a dataset or upload it.
   **Colab:** *Runtime > Change runtime type > GPU*, then upload the zip in the Files panel.
2. Sessions end; the **cache** (saved in the results folder) lets you resume by re-running the same cell.


In [ ]:
# ---- settings you may change -------------------------------------------------
import glob, os
ZIP_PATH   = (glob.glob('/kaggle/input/**/architecture-analyzer*.zip', recursive=True)
              + glob.glob('/content/architecture-analyzer*.zip')
              + glob.glob('./architecture-analyzer*.zip'))
ZIP_PATH   = ZIP_PATH[0] if ZIP_PATH else None
# Kaggle automatically UNZIPS uploaded zips, so the project is usually already a folder under /kaggle/input.
SRC_DIR    = None
if not ZIP_PATH:
    hits = glob.glob('/kaggle/input/**/backend/experiments/run_experiments.py', recursive=True)
    if hits:
        SRC_DIR = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))   # .../<project>/backend/experiments/x.py -> .../<project>
BASE_MODEL = 'qwen2.5:7b-instruct'   # check ollama.com/library for a current 7-8B *instruct* model; name may differ
NUM_CTX    = 4096                    # must exceed the longest prompt (~1,100 tokens) plus the reply
WORK       = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
print('zip:', ZIP_PATH, '| project folder:', SRC_DIR, '| work dir:', WORK)
assert ZIP_PATH or SRC_DIR, 'Project not found: add the Kaggle zip as a dataset (Input panel) and run this cell again. Listing /kaggle/input:\n' + '\n'.join(glob.glob('/kaggle/input/*/*')[:20])


## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'NO GPU DETECTED - turn on a GPU accelerator, or expect very slow runs'

## 2. Unpack the project and install its Python dependencies

In [ ]:
import subprocess, os, shutil
os.chdir(WORK)
shutil.rmtree('architecture-analyzer', ignore_errors=True)
if ZIP_PATH:
    subprocess.run(['unzip', '-q', '-o', ZIP_PATH, '-d', '.'], check=True)
else:
    shutil.copytree(SRC_DIR, 'architecture-analyzer', copy_function=shutil.copyfile)   # /kaggle/input is read-only; work on a copy
os.chdir(f'{WORK}/architecture-analyzer/backend')
!chmod -R u+w {WORK}/architecture-analyzer
!pip install -q -r requirements-dev.txt
!python -m pytest tests -q --ignore=tests/test_frontend_contract.py 2>&1 | tail -3
print('Expect: 452 passed, 5 skipped (frontend test excluded). If you see 429 or ~378 you are running an OLD zip.')


## 3. Install and start Ollama
If the install line fails, read its error: newer installers need `zstd` (`apt-get install -y zstd`), and some
platforms need `sudo`. The server must answer on port 11434 before continuing.

In [ ]:
!apt-get install -y -q zstd > /dev/null 2>&1 || true
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request, os
env = dict(os.environ, OLLAMA_NUM_PARALLEL='1', OLLAMA_CONTEXT_LENGTH=str(NUM_CTX))
server = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2); print('Ollama is up'); break
    except Exception:
        time.sleep(2)
else:
    print(open('/tmp/ollama.log').read()[-2000:]); raise SystemExit('Ollama did not start')

## 4. Pull the model and give it a bigger context window
Ollama silently truncates prompts longer than its context window. The Modelfile below sets `num_ctx`, and temperature 0
keeps answers repeatable.

In [ ]:
!ollama pull {BASE_MODEL}
open('/tmp/Modelfile', 'w').write(f'FROM {BASE_MODEL}\nPARAMETER num_ctx {NUM_CTX}\nPARAMETER temperature 0\n')
!ollama create arch-judge -f /tmp/Modelfile
!ollama show arch-judge | tee {WORK}/model_info.txt

## 5. Smoke test the model through the project's own client (checks JSON output and speed)

In [ ]:
import os, sys, json
os.environ.update(LLM_BASE_URL='http://localhost:11434/v1', LLM_MODEL='arch-judge', LLM_API_KEY='ollama')
sys.path.insert(0, f'{WORK}/architecture-analyzer/backend')
from analysis.semantic.llm_client import client_from_env
from analysis.semantic.llm_analyzer import parse_verdict_json, REVIEW_VERDICTS
c = client_from_env()
r = c.complete('You are a reviewer. Reply with ONLY JSON.',
               'Reply with {"verdict":"violation","confidence":0.9,"explanation":"test","evidence_used":[]}')
print('ok:', r.ok, '| latency %.1fs' % r.latency_s, '| tokens in/out:', r.input_tokens, r.output_tokens)
print('raw:', r.text[:200]); print('parsed:', parse_verdict_json(r.text, REVIEW_VERDICTS))

## 6. HARD TIER (current goal)
Harder-to-read ADRs plus two mixed-decision scenarios; same labels. See `docs/HARD_TIER.md`.

### 6a. Generate it and count the calls (no model calls yet)
Expect `Generated 72 cases`, M3 = 72 calls, M5 = 57.

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard --seed 7 --wording hard
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --dry-run --seeds 5 --methods deterministic_adr,full_hybrid,evidence_guided,generic_selective,no_history,no_intent

### 6b. Core run (about 125 new calls, roughly 5-15 minutes)
`--max-calls` stops safely if the session is about to hit a limit; **re-run the same cell to resume from the cache.**

In [ ]:
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --splits dev,test --methods static,deterministic_adr,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/hard --max-calls 400 --seeds 5

### 6c. Sanity check: did the model's answers parse?
Good: `parsed ok` almost equal to `calls`, errors 0. If many fail to parse, stop and report it.

In [ ]:
import pandas as pd
p = pd.read_csv('../results/hard/predictions.csv')
p = p[p.llm_called.astype(bool)]
print('calls:', len(p), '| parsed ok:', int(p.parsed_ok.sum()), '| errors:', int(p.error.notna().sum()),
      '| median latency: %.1fs' % p.latency_s.median())
print(open('../results/hard/summary.md').read()[:3500])

### 6d. OPTIONAL: M2, the LLM-only baseline (about 500 calls, 30-45 minutes). Skip if GPU quota is short.

In [ ]:
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --splits dev,test --methods llm_only --out ../results/hard_m2 --max-calls 600

## 7. M7 (extract-then-compute): dev check
M7 keeps M5's escalation but has the model *extract* facts from the ADR (without showing it any date) and lets code compare dates. Iterate on **dev only**. Re-running reuses cached answers, so only the new extraction calls cost anything (about 25-30 calls).

**Look for:** scenario D accuracy (M5 scored 0 of 6), `parsed ok` close to the number of calls, and few `fallback` rows.

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --splits dev --methods static,evidence_guided,extract_compute --out ../results/hard_m7_dev --max-calls 300 --bootstrap 200

In [ ]:
import pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120)
p = pd.read_csv('../results/hard_m7_dev/predictions.csv')
t = p[p.truth != 'conformant'].copy(); t['correct'] = t.predicted_violation == (t.truth == 'violation')
print(t[t.method.isin(['M5_evidence_guided','M7_extract_compute'])].groupby(['scenario','method']).correct.mean().unstack().round(2))
m7 = p[(p.method == 'M7_extract_compute') & p.llm_called.astype(bool)]
print('M7 calls:', len(m7), '| parsed ok:', int(m7.parsed_ok.sum()), '| fallbacks:', int((m7.outcome_source == 'fallback').sum()))
print(m7[m7.scenario.str.startswith('D')][['case_id','truth','verdict','computed']].to_string(index=False))

### 7b. Prompt x2 (only if the dev check showed the J regression)
Run this patch cell, then re-run cell 7 (the dev run) and the check cell below it. Only M7's ~30 extraction calls are repeated; everything else comes from the cache.

In [ ]:
# Patch for a session that still has prompt x1 (no restart or re-upload needed). Safe to run twice.
path = f'{WORK}/architecture-analyzer/backend/analysis/semantic/extract.py'
s = open(path).read()
if 'EXTRACT_VERSION = "x2"' in s:
    print('already x2')
else:
    old_yes = '    "yes"     the source module is named, or described unambiguously as the same code;\n'
    new_yes = ('    "yes"     the source module is named, or described unambiguously as the same code; or the introducing commit cites this ADR '
               '(see the history notes and the "Selected because" line). A decision that describes the code by its purpose rather than by file name '
               'still covers it when the commit cites it;\n')
    old_no = '    "no"      the decision is about a different module, or this module is expressly excluded;\n'
    new_no = '    "no"      the decision names or describes a different module, or expressly excludes this one;\n'
    assert old_yes in s and old_no in s and 'EXTRACT_VERSION = "x1"' in s, 'extract.py is not the expected x1 file'
    s = s.replace(old_yes, new_yes).replace(old_no, new_no).replace('EXTRACT_VERSION = "x1"', 'EXTRACT_VERSION = "x2"')
    open(path, 'w').write(s)
    print('patched to x2')
!grep -n 'EXTRACT_VERSION =' {path}


### 7c. Diagnostic: what did M7 extract for the mixed-decision cases (L, M)?

In [ ]:
j = p[(p.scenario.str[0].isin(['L','M'])) & (p.truth != 'conformant') & (p.method == 'M7_extract_compute')]
print(j[['case_id','truth','verdict','computed','extraction']].to_string(index=False))

### 7d. DEVELOPMENT seeds 21 and 22 (144 cases; prompt iteration happens here, NOT on the confirmation seeds 11-13)
About 200 new calls (roughly 10 minutes). `--max-calls` stops safely; re-run the cell to resume.

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s21 --seed 21 --wording hard
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s22 --seed 22 --wording hard
DEV = 'static,evidence_guided,extract_compute'
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s21 --splits dev,test --methods {DEV} --out ../results/dev_s21 --max-calls 300 --bootstrap 100
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s22 --splits dev,test --methods {DEV} --out ../results/dev_s22 --max-calls 300 --bootstrap 100

### 7e. Pool the development seeds and read them

In [ ]:
!python -m experiments.combine_runs --runs ../results/dev_s21 ../results/dev_s22 --out ../results/dev_all --bootstrap 500
import pandas as pd
pd.set_option('display.width', 220)
q = pd.read_csv('../results/dev_all/predictions.csv')
t = q[q.truth != 'conformant'].copy(); t['correct'] = t.predicted_violation == (t.truth == 'violation')
print(t[t.method.isin(['M5_evidence_guided','M7_extract_compute'])].groupby(['scenario','method']).correct.agg(['mean','sum','count']).unstack().round(2))
m7 = q[(q.method == 'M7_extract_compute') & q.llm_called.astype(bool)]
print('M7 calls:', len(m7), '| parsed ok:', int(m7.parsed_ok.sum()), '| fallbacks:', int((m7.outcome_source == 'fallback').sum()))

### 7f. Prompt x3 (fixes two flaws found in the L/M cases: the ADR's own date mistaken for a limit, and 'named' read as 'permitted')
Run the patch cell, then run x3 on the SAME development seeds (about 100 new calls; M5 comes from the cache), then compare.

In [ ]:
# Install prompt x3 (replaces extract.py). Safe to run twice; the checksum catches a damaged copy-paste.
import base64, zlib, hashlib
BLOB = """
eNqtWutz48aR/86/Yg6+KoFrEtm1k1wVbe1Gt5K9utIjJdG5bEk6LAgMKVggwGAASYzE/z2/7h48Sa6TuvCDBMyju6ff3QPHcU6eizwIi3Fxr9
NxmC1XZaFVrk2WlEWcpSqbq0DNk2Cx0JGK9EqnkU7DtXKXurjPInX+X0NvMJjea7XUgSlzrFrl2SzRS+XeB3mkiljnIxVlofndp6OrY396enLl
LSMVpJjDtu/efvfH8bt3hLNMCjOcqCeQgml1dHylFjnWzQMszM2gRh9ro+K0yLOoDIGwXKkiA5lRUOiR0o86X6uzs3NlSTRhRmS9pbP8UeFMhD
aYFzofh2WRzecqDIz2FB8ii3QyyHUgxBEJ2J3rsEjWFckpcD/qtNCyhrAq4lyQxwbQXadF2kzPgZzXzXOt/y7LnYpn8/hZxYZIjx9jY/mdBLOs
BM/SrMDMTBeglJi6XBWTwUDhR4dTiqg0DBr0Zbmw9OSv06ujj9NrNYdUwaWC4C8ysCEoiLVBAVlo2RbpUJCaAIcD0kJNP51eMwr5gR1lopW70v
kyLkZExX08oycmLoogNaOj4YhxxwKWFxsGnMTYRnzKVFt4LQwtXj3FxX0MyUNllhpvaZQ9KZcgMpgDox4wxpjCIAGwILciD7NVDABz8IgpKPRz
ARYT/I+Xxyf09/zPv0xPrnkW+hHFIbgB5ojcLD8qWljxWawug/w5LoZMHa/SYjHAx2RZSYp8aEEuIocexTOdAwo0hzbfZ0m0B8+Ih00arMw9+M
qoK/tobO7ADIIF1pqMJ1hV1X1gSBZg3IK5DDvIgQyqH4eM1lMnZBB2Rb2PtM486EjUj1SJ5zFGB3oieZl4AQlCwFlqhLcRbCvJVksov3LZpI8u
js4+X59e+38+O7qAWY8qRjy/U+P36vl72LM1JMgve7J8XZamUClZKvR7gENjomgfy4VsnnA0KFIRPMDkwFPQyRy32ubUekxWKhr6wWlNsXgJBc
5urCp/cAaMe6bV5cWJ+lupDUtBtL+1EA4ufgZ7ABsMDqCQQTUDlGuiZxZHZMc/BXECvwdJpFFCTAQM/bxK4hAEsLKCfzp/DIr4UU/IsZUpdM7o
AF4SnF8l65GMhokmlbZWiUGFw9eKxtIcpJkqDe8UxSG20N6HlHi4rcFBkkCGwaP4IFPkmCzzIKmsQLmPcZYEtGGI7WqVBCEr3wBmQa5IJwko+D
IHoFkQPnwZWbktSjZ9b+A4zoC1w/fnJWBr31fxcpXldHqoJsM2g4Ed+xXqVD3nWnaC0iBMAsAz1dZ6qF6h2S0003ZvsV4x02X8KAUzL1eEMkgG
siLA49rExtOPMRmS9hDNcr0ICjDP7juxU/9dQoi6t8/oZZDCnrwkWfphEpP6233wxB95YN8WMYf6WOdHf/WhUv5HxMLrkSJoq8HAOm3/LydX16
eXF+pQOc/fO4PB8cnHUxq4xsiLIypunJFyKj/ML6TsPhjtW39MY1aZnM3g7PT8dCoA0izVNNn4XD9L/Sz3JUr1pnYOctgE1MH15+vpyblvKSeK
HedzVrIngQsK7+H3SdNaUUailFEu2dSQTQPaT1yRWFW576WnCBJc2cXllLdHmpIC8q09lyjeq9ZgcQxrbGYjp+3WwbPCGLIW8pDp2hpH7SonKm
PwJpsXT7xenEtA/v0I/gGJwpJtMbzX4QNWVjkReNoiSLzd9eUvVx9PKn9B3iMi2yqaOFEa67umR1c/n0yrpVEsqcYI543De0oMNKyA0pec5sXr
GE9dtTIUF8wEWNLgqM1VDvbsrLrxHYcs81BbnHDLIIbyDo6cQb7QlGaMoVdWcM6kBsWuViRi9kK0yQHPyghla2qhU4TDRMJyrcxNJtDJSOBtsi
cDeYahJvOB4Ios4XBq2m4a1NMSA62KofIlRzHEDfhE2UrJRx5Eekx5HqDoZxolb/eDJaQ2pZ2EWI5TMKEQ1EI9Ap9/1ZJksR8mUcUp0QcVW5FE
4K4Ki2bLSGt2xhUnRQMrjgGgHQgoATQsWQiyCfdbMatmhsVZOYGGx2QaIXtl5EgJy+KcrIGjnLW0KJ7PkcSklPHSkev3irQRD+vnYLlKbLZXFE
QQmYLo39/KOGcSUzkDOPKg9QqJqGQheVYu7it7AAQEmrXOPbhv84S3LLW4t5VrgtBNMGqjsoUC2yudC2WIzfBFL8AdSyJliOTxCzr2tM7u4fXz
tKNUKp6TvAmFqXISFjjt5imof5gjvZPpECmLmsEZQRdWZb7KTJO+Vb6TaIYvojiOvKBSYqLANVqCMwYQkdZ0Cm1qAM61TjSnmzMdBjBVR5GWcX
pLjimBcqScW7C74qxJFJP5Y0ikQaPPvMYaX5tNDf+QbvVssU778eap00VK+S2rzNM9nZWylJI5zYYfMjGuKYEGGYKORjVBI3KVcHHkSXECOBlO
beBhtmuHnrlRgVjkcbijnPitIuKDmAOHP7Xrh6SK4wETY41nX5BULXWq9ZQ1YLsipZqo8UqMYxt6DVd+/wR04QMmbXW5D7IEa/UvQOYdFl4jHp
9rVgltncpLogYUShJUaN5TThhSNkxlqEiGeVXBpclsYXXIIa0oP+M3Pj8fHx+zhaUlHJP6mK3WdXn9g805gTwsE0acomA/FbCCvNI+ZVZAM0f1
0yXVbTstuMYiTsS240XOyQOnComG7TmoMCgmCmCrNyRMJm0wuNJmlaWiaygjzj6z5/ufa2Ru2YwUnSKg9d5Uik4GL04Q5X4cgYnOj3FUMYTMf8
1JUxKzWGoGvOfErgnATbx87YSs112R5bVx/aPGwuw5Xr+i2q87FfN1h1KNeqrhNEKkDXQELKEyKEiZu3zygGIIErMC9WWlHOa9s6ESYjDwz0+m
R/7x0fQE6WSuPZJHnGg3d/7v1rxxP0xuxm/u8DT8cPuGXgn1a4iEE3r0Wq4i/o+SobhR/h2cBpQAXgZrsedmcnjnvflPh1yRdzoEtkjP2ZBWPo
mG00OXhUWjQype8X8iNuU4V3qZ2RqqKWWROQZUp9jkkpoFaTGWeM36cnCMickBO2wz5A7Tuo4d0udqtW/guRmdtaGqASRaaNtWreKQWiHKLKnA
k5qZi3gYtq2ZDWuleFEt2uipY6bUhgUMMcIKKI5VS0VREgwryUokvJFHElLcc0JOnyrnNnW8XxHMXToaI+EHSnLBQ8+g/C1cWjWkiEkHaaTrgU
PhPe8cVpLIyZnnLAoA822PBTS5GCKhBdSFOLzAkbqyIRlAXW7mzjQuKD14wQavoOcNpD13riU4uVxyR0O7QELWxrljICCxGfTz4Em42xnykJDq
3B2q/zhszUzqVhYRcjN+d6e+PQRWpdzGG3ZwEqzN0Knw2pN14XhIH8EQd74V/AHqYKQOhPd273DjDNvCmR/8CHQqjg4dxhtHG+f9bXqgvm3Jjd
AMZQRjeOBS1O1bBDOTu2k848IzdkpYC+LH32Hhe6crTJvP+CipyTRd/WibyhPFnYdXtS1O6NknmwZV1br639Ppp8tfpnXZBt1faHKUVcNs0moc
SWcp4w4PdWRskzYtlzNqITPXkSRlZUI8hbYuVzYM2UqxVnWrtzXV7JrbAx4qJdcJHoM4oYaMM2yEWJnJRUZ9FrKtMUIefGhSZ3mU9ldbm1ynqS
Q9SwXtgoL3sLbSSl/SSqsDlmqe36ZnulU902JkZIZ7GqlNUYkjFZ3w2Z9Ojo4tOSZebBODNNlAJD41DIOE+g8vG6FG6iE2z9PtRNjum6hb54UJ
vjmwQwd3lb5tbp3GSPcwAVGJQrE/R7TwOSBQwGmJg8mojIqZYGsK2sEZsw0jVZVuqOFnqeQEp5UjdTsQXsP3HnW5bjZ9nZiWRCR7p1hDDXvq3B
ny/dKSU5T65KoNuEJP3QhmdM0rCEPoIBXP8gCGiFi8KMHdNjW8saamXjuu1jZK0YiL3AVlAOJJgOhmF5K74XCLFnIsueZyMqQ8ax8dXFyEfHUw
W9sai2jYibsHtIWXge7j/LVoqwX5QwWS97Bb87putRvzTC9y2YjlS3/HnXEfcdLrK3Z9HbxNwTKrCXS++Ub9tHXX5owaLjnXjeIiFggaT7TZXw
XF/Ua55MjgWGHQxXq1vYoGEYDaQKfc9ekBlVbQbwG1q3YBzWMcpG5dVRvoBZnwRt10RpACs9lu7ibdpZIucctm04LvfKTSJcizsiq0ucE6aS2x
UKjG8U0aQ+wF+XDH5byq8r1top32M2Rx9NU+Jup8E8+QcIF2/RgQa7K+F7cA71oCR35wsz/dWXqc8Cy9KrhzbFiSGdrzYN7ckXa33pVOUK/cOB
eS8mHCc/o4HTIbHOqC2wvcbN0Xm1xbEEsWuEA5n1ICansbbY5V4WVnwLcE1n5x2N/HJF1Jt5KvSLkPHEgroFVPccFanWjLHPmIZI7fqPH/+0cM
M4hRYtyoq/Kl+xgkJVT+KF13DdhSgpzBlDOUKTe3Znz3LZ3Jxx8sk5117lSlkJXr8GOTcZ7Vx1BdYNzQ5N2kcme8ioI2iXkruNOgpMRSPhlNbX
iQ5d5GL7/fDMf0/7vmf4fETvKw3A+7yNdbk3wChCR3efPuDgmiPH7XPH5fuWRpvqq/EMqTPM/y3YiEO5ywt0zDp5ujXonWyiPr/PGnOEfMbGtP
XJVZK3sVS5xEFa6RFWop/s0PDERqMiqUqU1KCSIZ+yN5sX5eKIX9Pj5hdxSLCcH0pCYSJRAlnlNQiHqC+vLlC8paOiZKXFS8ty/emw+3myGeMQ
V5ERyuYI8vp0dnZ02cY2gNNQ12L6a7x8J9O7KLvAVc5sp9ZyUCT1XckyLAhkANltVH4F7pSIX3xL+KVZoF0IrbwI4Vh4fKeXEmHfPGDEOnybeT
Qb/nVqGMOzOyA+7qXT2skwbHRvowvGqyY+O4vXGLCGmhM16b+XYNaQf7bLJAh77hnRMVIzeAnm/tmsFjP9ScC6X514LV4OkYEf2gp+AECR4uIo
iMS7saDF8xGyYX3jtOS92WCdcA8GOgGVJ3gWDExjL8ja1Nf0HaDofWBQKAZHB1TwpezhnCxHvz0m0a2W7TkAOuPA46MrGxVGoV1Vx0cieBMNsZ
ub78DaqluQZiazJsu23YN9CXDqCmK8fGL20AggTELg2NJERROwtn4SBLPlNW9cJZu13XsLFuv/H/3o5OC62JBl1myvywj63bWyOq6m3tKRFTXb
3XMDaDba/7b4mfNllflXLxr9wVJU+UhCHs/am51ue/6qr+0kxEnJUFNms+kFLfqINfUc9zI+8Afv6gvunltzLlD9VQKx0MpI1fyE75SupPcrdU
rK2nm1OhG/ENgl8Dco1O5hxLZlmWbDl0mvUsUdT5cWp6HCKvoYDvt8yejyzqUoHXttO9Z+mFjFoF5qQX/pm4Pp8QjKZEX/MxkuROq6QUIn6Oix
1fg2CEPpSzZHkQfPU5UPrIx2hHueebRqXv2P1WXeUtNjXkuU7Dk3YPe8cXNhW04W6ExO2q6/1VhLUsud9XI3x5vjmoO5t3m/59UgetGKkckr3V
VxE2OjCqSezeDHculJyqF1IZc1/CcFwVDWLsdRHfjO1N+/ayfi4QmQ/yAR2YYD8ooiSIxBG0PyZqONLqdvzLaC1Wi2hWFns+rQNg+9WSxZs98E
cq+y4nJm2qfjxssabtGHfcX3Q3/hP75Iajs+19a9vmplGYu6oplhLt05ySeMdAGQ3UgxphPwUJNW2d+npVJtfO5iZ76NQzuxWMpAHGcORpK/pg
5z0mZN4i+qXVJkJsATOWQeEON+qFCN6og45mvDQn7Kw+cIb/1tAQJMnAOv+zs3P7+fERfTf1d900gS9TWUqXlkobPLI6zeOU7lJH0qCblRG1LY
LqI9jHWD/JJ1gApdzzPwzZndXu3/fjFEf02ecjs+UPtybNN1wou4Nnv8gedGpY/BDq79++beVMHA5Cu5hfmh1YHG6BaZBbT21xf6U/xN+mtszN
rAC5hdir7ind7jdYo6+2ooYj7nVTDo94fPjWe9sm87B3liZvkmsTULC7HCPy5IJALjPwVqlrXUjsT78o9tDFoIXqjOpH37YmMNv7OI5cLtOEKX
moR/zsoR5sZ/j9zIlSaCxkavmlypg79LtVTmgBig22v92U0Ak/imTH6ednPFlh4RfQKdz1SyGfnB1ttYv4xQoAay1cOylvdnbTw2XB0res/JFa
tak/Tt45oPu9tW+qNfUAJsOALvuqGXnrocqDJ5/kXaOgkrSVVf4Dk5694g==
"""
raw = zlib.decompress(base64.b64decode(''.join(BLOB.split())))
assert hashlib.sha256(raw).hexdigest() == "53a8c462775aab98e76781280d0b4420dca425b06619adba9a689e74f6fdb2bb", 'checksum mismatch: the text was damaged when copied; copy the whole cell again'
path = f'{WORK}/architecture-analyzer/backend/analysis/semantic/extract.py'
open(path, 'wb').write(raw)
print('installed', len(raw), 'bytes')
!grep -n 'EXTRACT_VERSION =' {path}


In [ ]:
%cd {WORK}/architecture-analyzer/backend
M7 = 'static,evidence_guided,extract_compute'
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s21 --splits dev,test --methods {M7} --out ../results/dev_s21_x3 --max-calls 300 --bootstrap 100
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s22 --splits dev,test --methods {M7} --out ../results/dev_s22_x3 --max-calls 300 --bootstrap 100


In [ ]:
import pandas as pd
from experiments import metrics
pd.set_option('display.width', 220)
def load(tag, strip=''):
    d = pd.read_csv(f'../results/{tag}/predictions.csv'); d['case_id'] = tag.replace(strip, '') + ':' + d['case_id'].astype(str); return d
a = pd.concat([load('dev_s21'), load('dev_s22')]); a = a[a.method.isin(['M5_evidence_guided', 'M7_extract_compute'])].assign(method=lambda d: d.method.replace({'M7_extract_compute': 'M7_x2'}))
b = pd.concat([load('dev_s21_x3', '_x3'), load('dev_s22_x3', '_x3')]); b = b[b.method == 'M7_extract_compute'].assign(method='M7_x3')
allp = pd.concat([a, b]); allp['split'] = 'dev'
print(metrics.summarize(allp, n_boot=300)[['method', 'precision', 'recall', 'f1', 'f1_ci_low', 'f1_ci_high', 'llm_calls']].round(3).to_string(index=False))
t = allp[allp.truth != 'conformant'].copy(); t['correct'] = t.predicted_violation == (t.truth == 'violation')
print(t.groupby(['scenario', 'method']).correct.mean().unstack().round(2))
print(metrics.paired_bootstrap(allp, 'dev', ['M7_x3'], ['M7_x2'], 500).query("metric == 'f1'").round(3).to_string(index=False))
m = allp[(allp.method == 'M7_x3') & allp.llm_called.astype(bool)]
print('M7 x3 calls:', len(m), '| parsed ok:', int(m.parsed_ok.sum()), '| fallbacks:', int((m.outcome_source == 'fallback').sum()))


## 8. CONFIRMATION on fresh seeds (run ONLY after M7 is frozen and committed)
The confirmation protocol and its pass/fail criteria are in `docs/ANALYSIS_PLAN.md`. Three new benchmark seeds of the hard tier (216 cases), run once, then pooled. About 350 new calls (10-25 minutes). `--max-calls` stops safely; **re-run the same cell to resume**.

### 8a. Generate the seeds

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s11 --seed 11 --wording hard
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s12 --seed 12 --wording hard
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s13 --seed 13 --wording hard

### 8b. Run the frozen methods once per seed

In [ ]:
M = 'static,deterministic_adr,full_hybrid,evidence_guided,extract_compute'
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s11 --splits dev,test --methods {M} --out ../results/confirm_s11 --max-calls 400 --bootstrap 100
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s12 --splits dev,test --methods {M} --out ../results/confirm_s12 --max-calls 400 --bootstrap 100
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s13 --splits dev,test --methods {M} --out ../results/confirm_s13 --max-calls 400 --bootstrap 100

### 8c. Pool the seeds and read the verdict
The pooling tool refuses to combine runs that used a different model, prompt version or method set.

In [ ]:
!python -m experiments.combine_runs --runs ../results/confirm_s11 ../results/confirm_s12 ../results/confirm_s13 --out ../results/confirm_all
print(open('../results/confirm_all/summary.md').read()[:6000])

## 9. Package for download
Kaggle: *Output* tab. Colab: Files panel. Upload `results_m7.zip` back to the conversation. (The cache file is left out on purpose.)

In [ ]:
!cp {WORK}/model_info.txt ../results/hard/ 2>/dev/null || true
!cd .. && zip -qr {WORK}/results_m7.zip results -x '*.sqlite'
print('Download', f'{WORK}/results_m7.zip')

---
# LATER: confirmatory runs (do NOT run these yet)
These are the primary, minimal-prompt and held-out runs from `docs/ANALYSIS_PLAN.md`. Run them only after the policy and prompts are frozen and the plan is committed to Git.

## LATER 6. Generate the benchmark and count the calls (no model calls yet)

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated --seed 7
!python -m experiments.run_experiments --dry-run --seeds 5

## LATER 7. PILOT: one case per scenario (10 cases)
Look at the table, then at `predictions.csv`: is `parsed_ok` true? are `explanation`s sensible? is `latency_s` small?
If most calls fail to parse, fix that **before** spending the full run.

In [ ]:
!python -m experiments.run_experiments --limit-cases 10 --methods evidence_guided,full_hybrid --out ../results/pilot --bootstrap 100
import pandas as pd
p = pd.read_csv('../results/pilot/predictions.csv')
p = p[p.llm_called.astype(bool)]
print('calls:', len(p), '| parsed ok:', int(p.parsed_ok.sum()), '| errors:', int(p.error.notna().sum()),
      '| median latency: %.1fs' % p.latency_s.median())
p[['method','case_id','scenario','truth','verdict','predicted_violation','latency_s','explanation']]

## LATER 8. Dev split (iterate prompts HERE only, never on the test split)
Re-running after a prompt change re-calls only what changed (responses are cached).

In [ ]:
!python -m experiments.run_experiments --splits dev --methods static,full_hybrid,evidence_guided,no_history --out ../results/dev --bootstrap 200

## LATER 9. FULL run, primary setup (freeze prompts and policy first; `docs/ANALYSIS_PLAN.md` says what counts as support)
Without M2 first (99 distinct calls). `--max-calls` stops safely if the session is about to hit a limit; run the same cell again to resume from the cache.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --methods static,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/primary --max-calls 400 --seeds 5

### LATER 9b. M2 (LLM on every edge) — 420 calls, the expensive one. Run it ONCE, on the primary setup.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --methods llm_only --out ../results/primary_m2 --max-calls 600

## LATER 9c. Robustness runs (99 calls each; skip M2)
* **Minimal prompt**: same data, all scenario-specific guidance removed from the system prompts.
* **Held-out wording**: the same repositories with the ADR decision language rewritten. **Commit `docs/ANALYSIS_PLAN.md` to Git before this cell** so the plan provably came first.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --prompt-variant minimal --methods static,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/minimal --max-calls 400 --seeds 5

In [ ]:
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_heldout --seed 7 --wording heldout
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_heldout --splits dev,test --methods static,deterministic_adr,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/heldout --max-calls 400 --seeds 5

## LATER 10. Package everything for download

In [ ]:
!cp {WORK}/model_info.txt ../results/primary/ 2>/dev/null || true
!cd .. && zip -qr {WORK}/results.zip results
print('Download', f'{WORK}/results.zip', '(Kaggle: Output tab; Colab: Files panel).')
print(open('../results/primary/summary.md').read()[:3000])